# 02 — Feature Engineering

## Goal
Enrich the raw merged dataset with all engineered features before model training.
This notebook runs **once** and saves enriched artifacts consumed by all downstream notebooks.

## What this notebook does
1. Loads `train.parquet`, `val.parquet`, `test.parquet`
2. Concatenates all three and sorts by `TransactionDT` — required for no-leakage aggregations
3. Computes user aggregation features
4. Computes behavioral fingerprint features
5. Computes product profile features
6. Splits back into train / val / test using the `source` marker
7. Saves all enriched artifacts

## 3-Way Split Architecture
```
train.parquet (~354K, 60%, isFraud known)  → train_enriched.parquet + y_train.parquet
val.parquet   (~118K, 20%, isFraud known)  → val_enriched.parquet   + y_val.parquet   (early stopping + Optuna)
test.parquet  (~118K, 20%, isFraud known)  → test_enriched.parquet  + y_test.parquet  <- FROZEN TEST (touched once)
```

## Outputs
```
outputs/enriched/train_enriched.parquet  — enriched train features (no target)
outputs/enriched/val_enriched.parquet    — enriched val features   (no target)
outputs/enriched/test_enriched.parquet   — enriched frozen TEST features (no target)
outputs/enriched/y_train.parquet         — train target (isFraud)
outputs/enriched/y_val.parquet           — val target   (isFraud)
outputs/enriched/y_test.parquet          — frozen TEST target (isFraud)  <- FROZEN
```

## No-leakage guarantee
All features use `expanding().shift(1)` or `rolling(closed='left')` — each row sees
only chronologically prior transactions. `isFraud` is never used in feature computation.


## Step 0 — Setup

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

PROJECT_ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
for subdir in ['', 'src', 'v0']:
    p = os.path.join(PROJECT_ROOT_PATH, subdir)
    if p not in sys.path:
        sys.path.insert(0, p)

from config import PROJECT_ROOT, TIME_COL

print(f'Project root : {PROJECT_ROOT}')
print(f'Resolved root: {PROJECT_ROOT_PATH}')


Project root : C:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection
Resolved root: c:\Users\USER\Desktop\NOTEBOOKS\HIT\ML_Fraud_detection


## Step 1 — Load processed data

Load `train.parquet`, `val.parquet`, and `test.parquet`.
- **train** (~354K rows, 60%, `isFraud` known) — model training set
- **val**   (~118K rows, 20%, `isFraud` known) — early stopping + Optuna objective
- **test**  (~118K rows, 20%, `isFraud` known) — frozen TEST set (touched once at final eval)


In [2]:
from data_loader import load_processed

train_raw, val_raw, test_raw = load_processed(PROJECT_ROOT / 'data')

print(f'train_raw shape : {train_raw.shape}  | fraud rate: {train_raw["isFraud"].mean():.4%}')
print(f'val_raw   shape : {val_raw.shape}    | fraud rate: {val_raw["isFraud"].mean():.4%}')
print(f'test_raw  shape : {test_raw.shape}   | fraud rate: {test_raw["isFraud"].mean():.4%}  (frozen TEST)')

assert 'isFraud' in train_raw.columns, 'isFraud missing from train!'
assert 'isFraud' in val_raw.columns,   'isFraud missing from val!'
assert 'isFraud' in test_raw.columns,  'isFraud missing from test (frozen TEST)!'
print('\nColumn check passed ✓')


>> Loading processed parquet files (train + val + test)...
   train : 354,324 rows × 439 cols (937 MB)
   val   : 118,108 rows × 439 cols (314 MB)  <- early stopping
   test  : 118,108 rows × 439 cols (314 MB)  <- frozen TEST
   Dtypes: {dtype('float32'): np.int64(399), dtype('O'): np.int64(32), dtype('int32'): np.int64(4), dtype('int64'): np.int64(2), dtype('int8'): np.int64(1), dtype('int16'): np.int64(1)}

   Train preview (top 7 rows):


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_36,id_37,id_38,DeviceType,DeviceInfo,tx_day,tx_dow,tx_hour,tx_dom,DeviceType_filled
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,None,None,None,None,None,1,1,0,2,No device info
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,None,None,None,None,None,1,1,0,2,No device info
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,None,None,None,None,None,1,1,0,2,No device info
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,None,None,None,None,None,1,1,0,2,No device info
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,1,0,2,mobile
5,2987005,0,86510,49.0,W,5937,555.0,150.0,visa,226.0,...,None,None,None,None,None,1,1,0,2,No device info
6,2987006,0,86522,159.0,W,12308,360.0,150.0,visa,166.0,...,None,None,None,None,None,1,1,0,2,No device info


train_raw shape : (354324, 439)  | fraud rate: 3.3833%
val_raw   shape : (118108, 439)    | fraud rate: 3.9041%
test_raw  shape : (118108, 439)   | fraud rate: 3.4409%  (frozen TEST)

Column check passed ✓


## Step 2 — Concatenate train + val + test, sort by time

All feature engineering runs on the **full dataset** (all three sets concatenated).
Val and test rows must see their prior history from train — otherwise they have
zero aggregation history and predictions misrepresent production behavior.

**No-leakage guarantee**: `isFraud` is never used in any feature computation.
The `source` marker identifies each row's origin for split-back.


In [3]:
# Add source marker BEFORE concat — the only reliable split key after concat.
# WHY 'source' over binary 'is_train': with three sets, a binary flag is ambiguous.
train_raw = train_raw.copy()
val_raw   = val_raw.copy()
test_raw  = test_raw.copy()
train_raw['source'] = 'train'
val_raw['source']   = 'val'
test_raw['source']  = 'test'

full = pd.concat([train_raw, val_raw, test_raw], ignore_index=True)
full = full.sort_values('TransactionDT').reset_index(drop=True)

assert (full['source'] == 'train').sum() == len(train_raw), 'Train count mismatch!'
assert (full['source'] == 'val').sum()   == len(val_raw),   'Val count mismatch!'
assert (full['source'] == 'test').sum()  == len(test_raw),  'Test count mismatch!'
# All three splits have isFraud known — no NaN expected
assert full['isFraud'].isna().sum() == 0, 'Unexpected NaN in isFraud — all splits must have labels!'

print(f'Full dataset shape : {full.shape}')
print(f'  train rows : {(full["source"]=="train").sum():,}  (60%)')
print(f'  val rows   : {(full["source"]=="val").sum():,}  (20% — early stopping + Optuna)')
print(f'  test rows  : {(full["source"]=="test").sum():,}  (20% — frozen TEST)')
print(f'TransactionDT range: {full["TransactionDT"].min()} -> {full["TransactionDT"].max()}')


Full dataset shape : (590540, 440)
  train rows : 354,324  (60%)
  val rows   : 118,108  (20% — early stopping + Optuna)
  test rows  : 118,108  (20% — frozen TEST)
TransactionDT range: 86400 -> 15811131


## Step 3 — User aggregation features (18 features)

Computes per-user (card1 + addr1) behavior statistics, strictly before each transaction:
- **Cumulative stats**: tx_count, amt_mean/std/min/max, tx_amt_ratio, time_since_last_tx, delta_amt
- **Email instability**: nunique_P/R_email, is_new_P/R_email, is_same_email_domain
- **Device instability**: nunique_device, is_new_device
- **Velocity**: tx_count_last_3d / 7d / 30d

In [4]:
from preproc_agg import compute_user_aggregations

# compute_user_aggregations returns (df_with_features, list_of_new_col_names).
# It internally re-sorts by TransactionDT and resets index — full remains aligned.
full, agg_feature_cols = compute_user_aggregations(full, verbose=True)

print(f"\nAggregation features added ({len(agg_feature_cols)}): {agg_feature_cols}")
print(f"Full shape after aggregations: {full.shape}")

FEATURE ENGINEERING — User Behavior Aggregations
   Group key:    ['card1', 'addr1']
   Shape before: (590540, 440)

>> Feature Engineering: Cumulative Aggregations...
>> Feature Engineering: Email Instability...
>> Feature Engineering: Device Instability...
>> Feature Engineering: Rolling Window Velocity...

   Shape after:  (590540, 458)
   New features (18):
     + tx_count
     + tx_amt_mean
     + tx_amt_std
     + tx_amt_min
     + tx_amt_max
     + tx_amt_ratio
     + time_since_last_tx
     + delta_amt
     + nunique_P_email
     + is_new_P_email
     + nunique_R_email
     + is_new_R_email
     + is_same_email_domain
     + nunique_device
     + is_new_device
     + tx_count_last_3d
     + tx_count_last_7d
     + tx_count_last_30d

Aggregation features added (18): ['tx_count', 'tx_amt_mean', 'tx_amt_std', 'tx_amt_min', 'tx_amt_max', 'tx_amt_ratio', 'time_since_last_tx', 'delta_amt', 'nunique_P_email', 'is_new_P_email', 'nunique_R_email', 'is_new_R_email', 'is_same_email_domain

## Step 4 — Behavioral fingerprint features (4 features)

Captures how much each transaction deviates from the **personal norm** of that user:
- `amt_vs_personal_median` — current amount / user's historical median amount
- `amt_z_score`           — (current amount − user mean) / user std
- `hour_vs_typical`       — deviation from user's typical transaction hour
- `uid_time_entropy`      — entropy of user's historical amount distribution (low = bot-like)

In [5]:
from preproc_behavioral import compute_behavioral_features

full, behavioral_feature_cols = compute_behavioral_features(full, verbose=True)

print(f"\nBehavioral features added ({len(behavioral_feature_cols)}): {behavioral_feature_cols}")
print(f"Full shape after behavioral features: {full.shape}")

FEATURE ENGINEERING — Behavioral Fingerprint
   Group key    : ['card1', 'addr1']
   Shape before : (590540, 458)

   Computing amt_vs_personal_median ...
   Computing amt_z_score ...
   Computing hour_vs_typical ...
   Computing uid_time_entropy (entropy loop — may take ~1-2 min) ...

   Shape after  : (590540, 462)
   New features : ['amt_vs_personal_median', 'amt_z_score', 'hour_vs_typical', 'uid_time_entropy']

   NaN check: 0 unexpected NaN in new features ✓

Behavioral features added (4): ['amt_vs_personal_median', 'amt_z_score', 'hour_vs_typical', 'uid_time_entropy']
Full shape after behavioral features: (590540, 462)


## Step 5 — Product profile features (2 features)

Captures how each transaction relates to the **product-level norm**:
- `is_new_product`         — 1 if user is buying this ProductCD for the first time
- `amt_vs_product_typical` — current amount / median amount for this ProductCD

In [6]:
from preproc_product import compute_product_features

full, product_feature_cols = compute_product_features(full, verbose=True)

print(f"\nProduct features added ({len(product_feature_cols)}): {product_feature_cols}")
print(f"Full shape after product features: {full.shape}")

FEATURE ENGINEERING — Product Profile
   Group key    : ['card1', 'addr1']
   Product col  : ProductCD
   Shape before : (590540, 462)

   Computing is_new_product ...

   Computing amt_vs_product_median ...

   Shape after  : (590540, 464)
   is_new_product distribution    : {0: 539651, 1: 50889}
   First-time product purchases   : 8.6% of all transactions
   amt_vs_product_median — NaN    : 0
   amt_vs_product_median — filled : 1 rows with -1
   NaN check: 0 unexpected NaN ✓

Product features added (2): ['is_new_product', 'amt_vs_product_median']
Full shape after product features: (590540, 464)


## Step 6 — Split back into train / val / test + extract targets

Uses the `source` marker set before concat — never modified during feature engineering.

- `train_enriched` — 60% train (~354K rows); used for model training
- `val_enriched`   — 20% val (~118K rows); used for early stopping + Optuna
- `test_enriched`  — 20% frozen TEST (~118K rows); touched once at final eval in notebook 04

WHY extract targets here (after all feature engineering):
`compute_user_aggregations()` internally calls `sort_values().reset_index(drop=True)` —
the original index is replaced. Targets must be extracted from `full` AFTER this reset
to guarantee index alignment between feature DataFrames and target Series.


In [7]:
train_enriched = full[full['source'] == 'train'].copy()
val_enriched   = full[full['source'] == 'val'].copy()
test_enriched  = full[full['source'] == 'test'].copy()

# Extract targets AFTER feature engineering — index was reset inside compute_user_aggregations.
# WHY after split back: each y is aligned with its enriched DataFrame by construction.
y_train = train_enriched['isFraud'].astype(int)
y_val   = val_enriched['isFraud'].astype(int)
y_test  = test_enriched['isFraud'].astype(int)

# ── Size assertions ───────────────────────────────────────────────────────
assert len(train_enriched) == len(train_raw), f'Train size mismatch: {len(train_enriched)} vs {len(train_raw)}'
assert len(val_enriched)   == len(val_raw),   f'Val size mismatch: {len(val_enriched)} vs {len(val_raw)}'
assert len(test_enriched)  == len(test_raw),  f'Test size mismatch: {len(test_enriched)} vs {len(test_raw)}'

# ── Index alignment assertions ────────────────────────────────────────────
# WHY: y and X must have identical index — saved separately, loaded independently in pipeline
assert y_train.index.equals(train_enriched.index), 'y_train / train_enriched index mismatch!'
assert y_val.index.equals(val_enriched.index),     'y_val / val_enriched index mismatch!'
assert y_test.index.equals(test_enriched.index),   'y_test / test_enriched index mismatch!'

# ── Label value assertions ────────────────────────────────────────────────
assert set(y_train.unique()).issubset({0, 1}), f'Unexpected y_train values: {y_train.unique()}'
assert set(y_val.unique()).issubset({0, 1}),   f'Unexpected y_val values: {y_val.unique()}'
assert set(y_test.unique()).issubset({0, 1}),  f'Unexpected y_test values: {y_test.unique()}'

print('All alignment assertions passed ✓')
print(f'train_enriched : {train_enriched.shape}  | fraud rate: {y_train.mean():.4%}')
print(f'val_enriched   : {val_enriched.shape}    | fraud rate: {y_val.mean():.4%}')
print(f'test_enriched  : {test_enriched.shape}   | fraud rate: {y_test.mean():.4%}  (frozen TEST)')


All alignment assertions passed ✓
train_enriched : (354324, 464)  | fraud rate: 3.3833%
val_enriched   : (118108, 464)    | fraud rate: 3.9041%
test_enriched  : (118108, 464)   | fraud rate: 3.4409%  (frozen TEST)


## Step 7 — Drop markers + save all enriched artifacts

Drops `isFraud` and `source` from all three feature DataFrames — both were markers only.
Targets `y_train`, `y_val`, `y_test` are already extracted and saved separately.

WHY drop `isFraud` from feature DataFrames: X files must never contain the target —
model training would silently use it as a feature, causing 100% leakage.
WHY drop `source`: it was a split-back marker only, not a real feature.


In [8]:
enriched_dir = PROJECT_ROOT / 'outputs' / 'enriched'
enriched_dir.mkdir(parents=True, exist_ok=True)

# Drop markers from all three feature DataFrames
# WHY: isFraud and source were markers only — must not appear in X files
marker_cols = ['isFraud', 'source']
train_enriched = train_enriched.drop(columns=[c for c in marker_cols if c in train_enriched.columns])
val_enriched   = val_enriched.drop(  columns=[c for c in marker_cols if c in val_enriched.columns])
test_enriched  = test_enriched.drop( columns=[c for c in marker_cols if c in test_enriched.columns])

# Save feature DataFrames — index=True preserves alignment with y files
train_enriched.to_parquet(enriched_dir / 'train_enriched.parquet', index=True)
val_enriched.to_parquet(  enriched_dir / 'val_enriched.parquet',   index=True)
test_enriched.to_parquet( enriched_dir / 'test_enriched.parquet',  index=True)

# Save targets — index=True keeps them aligned with X DataFrames
y_train.to_frame().to_parquet(enriched_dir / 'y_train.parquet', index=True)
y_val.to_frame().to_parquet(  enriched_dir / 'y_val.parquet',   index=True)
y_test.to_frame().to_parquet( enriched_dir / 'y_test.parquet',  index=True)

print('Saved:')
for name in ['train_enriched.parquet', 'val_enriched.parquet', 'test_enriched.parquet',
             'y_train.parquet', 'y_val.parquet', 'y_test.parquet']:
    path = enriched_dir / name
    size_mb = path.stat().st_size / 1024**2 if path.exists() else 0
    tag = '  <- FROZEN TEST' if 'test' in name else ''
    print(f'  {name:<35} {size_mb:.0f} MB{tag}')
print(f'\ntrain_enriched : {train_enriched.shape}')
print(f'val_enriched   : {val_enriched.shape}')
print(f'test_enriched  : {test_enriched.shape}  (frozen TEST)')


Saved:
  train_enriched.parquet              66 MB
  val_enriched.parquet                22 MB
  test_enriched.parquet               23 MB  <- FROZEN TEST
  y_train.parquet                     2 MB
  y_val.parquet                       1 MB
  y_test.parquet                      1 MB  <- FROZEN TEST

train_enriched : (354324, 462)
val_enriched   : (118108, 462)
test_enriched  : (118108, 462)  (frozen TEST)


## Step 8 — Feature engineering summary

Quick overview of all new features added in this notebook.

In [9]:
all_new_cols = agg_feature_cols + behavioral_feature_cols + product_feature_cols

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"  Raw features (v0 baseline)    : {len(train_raw.columns) - 3}")  # minus isFraud, TransactionID, TransactionDT
print(f"  Aggregation features (Step 3) : {len(agg_feature_cols)}")
print(f"  Behavioral features  (Step 4) : {len(behavioral_feature_cols)}")
print(f"  Product features     (Step 5) : {len(product_feature_cols)}")
print(f"  Total new features            : {len(all_new_cols)}")
print(f"  Total features in train_enriched (incl. ID/DT cols): {train_enriched.shape[1]}")
print()

# Check for NaN in new features — unexpected NaN outside of first-transaction fill
new_feat_nan = train_enriched[all_new_cols].isna().sum()
new_feat_nan = new_feat_nan[new_feat_nan > 0]
if len(new_feat_nan) == 0:
    print("  NaN check: no unexpected NaN in new features ✓")
else:
    print(f"  NaN in new features (first-transaction fill expected):\n{new_feat_nan}")

FEATURE ENGINEERING SUMMARY
  Raw features (v0 baseline)    : 437
  Aggregation features (Step 3) : 18
  Behavioral features  (Step 4) : 4
  Product features     (Step 5) : 2
  Total new features            : 24
  Total features in train_enriched (incl. ID/DT cols): 462

  NaN in new features (first-transaction fill expected):
tx_count    38927
dtype: int64


---
---
# Summary

| | Rows | Columns | Fraud rate |
|---|---|---|---|
| `train_enriched` | 354,324 (60%) | 461 | 3.38% |
| `val_enriched` | 118,108 (20%) | 461 | 3.90% |
| `test_enriched` | 118,108 (20%) | 461 | 3.44% |

**New features added:** 
* 23 total:
    * 18 aggregation
    * 4 behavioral 
    * 1 product

**NaN in new features:** 
* `tx_count` only — 38,927 rows (first transaction per UID, expected). 
* Filled with `-1` in notebook 03.

**Saved to** `outputs/enriched/`:
- `train_enriched.parquet` (65 MB), `y_train.parquet` (2 MB)
- `val_enriched.parquet` (22 MB), `y_val.parquet` (1 MB)
- `test_enriched.parquet` (23 MB), `y_test.parquet` (1 MB) ← frozen TEST
